# Assignment: CNN Classification and Segmentation

Name:

UID:

Please submit:
- A PDF containing all outputs after running the notebook
- Your completed `.ipynb` file

I understand the course academic integrity policy. Please sign your name here:

## Overview

In this assignment, you will work with two core computer vision tasks:

1. Image classification using a convolutional neural network trained on CIFAR-10.
2. Image segmentation using the Segment Anything Model, SAM.

Classification setting:
- Batch size: 4
- Epochs: 5
- Optimizer: Adam
- Learning rate: 0.001

Expected classification accuracy:
- A correct implementation should usually reach around 45% to 60% test accuracy with this small batch size and short training time.
- Results may vary slightly depending on runtime, hardware, and random initialization.

In [ ]:
import os
import sys
import random
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# Part 1: Image Classification with CNN (80 pts, including bonus points)

## Goal

In this part, you will train a convolutional neural network on CIFAR-10.

You will:

1. Load CIFAR-10.
2. Display sample training images.
3. Define a CNN.
4. Train the CNN.
5. Evaluate test accuracy.
6. Compute per-class accuracy and average class accuracy.
7. Plot training loss and test accuracy.

## 1.1 Load CIFAR-10



In [ ]:
!pip install datasets -q

In [ ]:
from datasets import load_dataset

batch_size = 4

transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5),
                         (0.5, 0.5, 0.5))
])

class HFCIFAR10Dataset(Dataset):
    def __init__(self, split, transform=None):
        self.dataset = load_dataset("uoft-cs/cifar10", split=split)
        self.transform = transform

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        item = self.dataset[idx]
        image = item["img"]
        label = item["label"]

        if self.transform:
            image = self.transform(image)

        return image, label

trainset = HFCIFAR10Dataset("train", transform=transform)
testset = HFCIFAR10Dataset("test", transform=transform)

trainloader = DataLoader(trainset, batch_size=batch_size, shuffle=True, num_workers=2)
testloader = DataLoader(testset, batch_size=batch_size, shuffle=False, num_workers=2)

classes = ("plane", "car", "bird", "cat", "deer",
           "dog", "frog", "horse", "ship", "truck")

print("Training images:", len(trainset))
print("Testing images:", len(testset))
print("Batch size:", batch_size)

## 1.2 Display Training Images

In [ ]:
def imshow(img):
    img = img / 2 + 0.5
    npimg = img.numpy()
    plt.figure(figsize=(8, 4))
    plt.imshow(np.transpose(npimg, (1, 2, 0)))
    plt.axis("off")
    plt.show()

dataiter = iter(trainloader)
images, labels = next(dataiter)

imshow(torchvision.utils.make_grid(images))
print("Labels:", " ".join(classes[labels[j]] for j in range(len(labels))))

## 1.3 Define a Simple CNN (20 pts)

The target architecture is:

```text
Conv -> ReLU -> MaxPool
Conv -> ReLU -> MaxPool
Flatten
Linear -> ReLU -> Linear
```

This is a lightweight CNN designed to run faster in Colab.

In [ ]:
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        """
          Add layers to your neural net.
          Tutorial for reference: https://pytorch.org/tutorials/beginner/introyt/trainingyt.html
        """

        # TODO:
        # Define two convolution layers.
        # The first should take RGB images as input.
        self.conv1 = None
        self.conv2 = None

        # TODO:
        # Define a max pooling layer.
        self.pool = None

        # TODO:
        # Define two fully connected layers.
        self.fc1 = None
        self.fc2 = None

    def forward(self, x):
        """
          Forward pass:
            Apply layers you defined in __init__() to RGB input
            layers: Conv2d --> ReLU --> MaxPool2d --> Conv2d --> ReLU --> MaxPool2d --> Flatten --> Linear --> ReLU --> Linear --> ReLU --> Linear
        """
        # TODO:
        # Apply conv -> relu -> pool twice.
        # Flatten the tensor.
        # Apply two fully connected layers.
        pass

net = Net().to(device)
print(net)

## 1.4 Define Loss and Optimizer (5 pts)

Use cross-entropy loss and Adam optimizer.

The learning rate is fixed at 0.001.

In [ ]:
# TODO YOUR CODE HERE: Add loss function (hint: use nn.function_name())
criterion = None

# TODO YOUR CODE HERE: Add optimizer (hint: use Adam)
optimizer = None

## 1.5 Train the Network (10 pts)

Train for 5 epochs. During training, record:

- average training loss per epoch
- test accuracy per epoch
- You should at least get 60% accuracy

## Extra Credit (5 pts)
- 5 pts bonus if you get more than 65% accuracy

In [ ]:
epochs = 5

train_losses = []
test_accuracies = []

for epoch in range(epochs):
    net.train()
    running_loss = 0.0
    num_batches = 0

    for images, labels in trainloader:
        images = images.to(device)
        labels = labels.to(device)

        # TODO:
        # 1. Clear gradients.
        # 2. Run forward pass.
        # 3. Compute loss.
        # 4. Run backward pass.
        # 5. Update weights.

        running_loss += loss.item()
        num_batches += 1

    avg_loss = running_loss / num_batches
    train_losses.append(avg_loss)

    net.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in testloader:
            images = images.to(device)
            labels = labels.to(device)

            # TODO:
            # Run prediction and update correct / total.

    test_acc = 100 * correct / total
    test_accuracies.append(test_acc)

    print(f"Epoch {epoch + 1:02d}/{epochs}, Loss: {avg_loss:.4f}, Test Accuracy: {test_acc:.2f}%")

## 1.6 Final Per-Class Accuracy and Average Class Accuracy (10 pts)

Compute accuracy for each CIFAR-10 class.

Then compute average class accuracy:

```text
average class accuracy = mean(per-class accuracies)
```

In [ ]:
correct_pred = {classname: 0 for classname in classes}
total_pred = {classname: 0 for classname in classes}

net.eval()

with torch.no_grad():
    for images, labels in testloader:
        images = images.to(device)
        labels = labels.to(device)

        # TODO:
        # 1. Compute model outputs.
        # 2. Convert outputs into predicted labels.
        # 3. Update correct_pred and total_pred for each class.

class_accuracies = []

for classname in classes:
    accuracy = 100 * correct_pred[classname] / total_pred[classname]
    class_accuracies.append(accuracy)
    print(f"Accuracy for class {classname:5s}: {accuracy:.1f}%")

# TODO:
# Compute average_class_accuracy as the mean of class_accuracies.
average_class_accuracy = None

print(f"Average class accuracy: {average_class_accuracy:.1f}%")

## 1.7 Plots (5 pts)

In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(range(1, len(train_losses) + 1), train_losses, marker="o")
plt.xlabel("Epoch")
plt.ylabel("Average training loss")
plt.title("Training Loss over Epochs")
plt.grid(True)
plt.show()

plt.figure(figsize=(6, 4))
plt.plot(range(1, len(test_accuracies) + 1), test_accuracies, marker="o")
plt.xlabel("Epoch")
plt.ylabel("Test accuracy (%)")
plt.title("Test Accuracy over Epochs")
plt.grid(True)
plt.show()

## Part 1 Write-up (20 pts)

Answer the following questions.

1. What is your final test accuracy?
2. What is your average class accuracy?
3. Which CIFAR-10 classes are easiest or hardest for your model?
4. Why might average class accuracy be useful in addition to overall test accuracy?

## Extra Credits (5 pt)
Run VGG with pre-trained weights in this [colab](https://colab.research.google.com/github/pytorch/pytorch.github.io/blob/master/assets/hub/pytorch_vision_vgg.ipynb#scrollTo=daily-wayne). Test any two images of your choice to your model and to VGG model and show accuracy (images must include objects from CIFAR10 classes). Discuss which model performs better and why.

In [ ]:
# YOUR CODE HERE

# Part 2: Image Segmentation with SAM (30 pts)

## Goal

In this part, you will use the Segment Anything Model, SAM, for image segmentation.

You will:

1. Load SAM.
2. Run SAM on a provided example image.
3. Run SAM on two images of your choice.
4. Analyze one failure case.

To make this assignment easier to run, we use the smaller `vit_b` SAM model.

In [ ]:
%%capture
!pip install opencv-python matplotlib
!pip install 'git+https://github.com/facebookresearch/segment-anything.git'

!mkdir -p images
!wget -q -P images https://raw.githubusercontent.com/facebookresearch/segment-anything/main/notebooks/images/truck.jpg
!wget -q -P images https://raw.githubusercontent.com/facebookresearch/segment-anything/main/notebooks/images/groceries.jpg

!wget -q https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth

In [ ]:
import cv2
from PIL import Image

def show_mask(mask, ax, random_color=False):
    if random_color:
        color = np.concatenate([np.random.random(3), np.array([0.6])], axis=0)
    else:
        color = np.array([30 / 255, 144 / 255, 255 / 255, 0.6])

    h, w = mask.shape[-2:]
    mask_image = mask.reshape(h, w, 1) * color.reshape(1, 1, -1)
    ax.imshow(mask_image)

def show_points(coords, labels, ax, marker_size=375):
    pos_points = coords[labels == 1]
    neg_points = coords[labels == 0]

    ax.scatter(pos_points[:, 0], pos_points[:, 1],
               color="green", marker="*", s=marker_size,
               edgecolor="white", linewidth=1.25)

    ax.scatter(neg_points[:, 0], neg_points[:, 1],
               color="red", marker="*", s=marker_size,
               edgecolor="white", linewidth=1.25)

def load_rgb_image(path):
    image = cv2.imread(path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    return image

In [ ]:
from segment_anything import sam_model_registry, SamPredictor

sam_checkpoint = "sam_vit_b_01ec64.pth"
model_type = "vit_b"

sam = sam_model_registry[model_type](checkpoint=sam_checkpoint)
sam.to(device=device)

predictor = SamPredictor(sam)
print("SAM loaded.")

## 2.1 Example: Segment an Object in the Truck Image

In [ ]:
image = load_rgb_image("images/truck.jpg")
predictor.set_image(image)

input_point = np.array([[500, 375]])
input_label = np.array([1])

masks, scores, logits = predictor.predict(
    point_coords=input_point,
    point_labels=input_label,
    multimask_output=True,
)

best_mask_index = np.argmax(scores)

plt.figure(figsize=(8, 8))
plt.imshow(image)
show_mask(masks[best_mask_index], plt.gca())
show_points(input_point, input_label, plt.gca())
plt.title(f"Best mask, score = {scores[best_mask_index]:.3f}")
plt.axis("off")
plt.show()

## 2.2 Your Own Images (10 pts)

Run SAM on two images of your choice.

For each image:

1. Display the original image.
2. Choose one foreground point.
3. Display one segmentation mask.
4. Briefly explain whether the mask is good or bad.

In [ ]:
# TODO:
# Replace the image paths and input points below with your own choices.
# You may upload images to Colab and use their file paths.

image_path_1 = "images/truck.jpg"
input_point_1 = np.array([[500, 375]])

image_path_2 = "images/groceries.jpg"
input_point_2 = np.array([[320, 250]])

for path, point in [(image_path_1, input_point_1), (image_path_2, input_point_2)]:
    image = load_rgb_image(path)
    predictor.set_image(image)

    input_point = point
    input_label = np.array([1])

    # TODO:
    # Run predictor.predict.
    # Select one mask.
    # Display the image with the selected mask and point.

## 2.3 Failure Case (5 pts)

Find one case where SAM does not segment the object well.

Examples:
- The mask includes too much background.
- The mask misses part of the object.
- The object is very small.
- The object is occluded.
- The scene has many overlapping objects.

In [ ]:
# TODO:
# Run SAM on one failure case.
# Display the image, selected point, and mask.
# Then explain why the result may have failed.

## Part 2 Write-up (15 pts)

Answer the following questions.

1. What types of images or objects does SAM segment well?
2. What types of images or objects are difficult for SAM?
3. How does changing the input point affect the result?

# Grading Rubric

Accuracy guidance for Part 1:
- Below 45%: likely implementation or training issue
- 45% to 55%: acceptable for this small model and short training setting
- 55% to 60%: good result
- Above 60%: strong result for this lightweight CNN